In [1]:
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image, HRFlowable
from reportlab.lib import colors
from PIL import Image as PILImage

#-----------------------------------------
#            Core Functions
#-----------------------------------------
def convert_image_to_bw(image_path):
    """
    Convert an image to black and white.
    """
    img = PILImage.open(image_path).convert('L')
    bw_image_path = "bw_" + image_path.split('/')[-1]
    img.save(bw_image_path)
    return bw_image_path

def create_invoice(invoice_number, invoice_date, service_date, recipient_name, recipient_phone, recipient_email, 
                   dj_name, time_slot, rate, duration_hours=1, additional_amount=0, service_fee=0,
                   due_date="", venmo_handle="", cashapp_handle="", payment_image_path="", logo_path="",
                   speaker_rental=0, advance_payment=0, final_payment_due_date=""):
    """
    Create a professional invoice PDF with enhanced features:
      - Calculates total service cost from an hourly rate and duration.
      - Optionally includes a speaker rental fee as a separate line item.
      - Supports a split payment plan (with advance and final payment sections).
      
    Parameters:
        invoice_number         (str)   : Invoice identifier.
        invoice_date           (str)   : Date of invoice creation.
        service_date           (str)   : Date of service.
        recipient_name         (str)   : Billing recipient name.
        recipient_phone        (str)   : Billing recipient phone.
        recipient_email        (str)   : Billing recipient email.
        dj_name                (str)   : DJ or service provider name.
        time_slot              (str)   : Time slot of the service.
        rate                   (str/float) : Hourly rate.
        duration_hours         (float) : Total number of service hours.
        additional_amount      (float) : Any additional charges.
        service_fee            (float) : Service fee.
        due_date               (str)   : Due date for the advance payment.
        venmo_handle           (str)   : Venmo payment handle.
        cashapp_handle         (str)   : Cash App payment handle.
        payment_image_path     (str)   : Path for the payment method image.
        logo_path              (str)   : Path for the company logo image.
        speaker_rental         (float) : (Optional) Speaker rental fee.
        advance_payment        (float) : (Optional) Advance payment amount.
        final_payment_due_date (str)   : (Optional) Due date for the remaining balance.
        
    The PDF invoice is saved in the '_1a_dta_invoices' folder.
    """
    # Calculate the base service cost based on the hourly rate and service duration.
    base_rate = float(rate) * duration_hours
    # Calculate the grand total including any additional amounts and speaker rental fees.
    total_due_amount = base_rate + additional_amount + service_fee + speaker_rental
    pdf_filename = f"invoice_{invoice_number}.pdf"
    doc = SimpleDocTemplate(f"_1a_dta_invoices/{pdf_filename}", pagesize=letter)
    
    styles = getSampleStyleSheet()
    styles.add(ParagraphStyle(name='TitleCenter', fontSize=18, alignment=1, spaceAfter=12, fontName="Helvetica-Bold"))
    styles.add(ParagraphStyle(name='Header', fontSize=12, fontName="Helvetica-Bold", spaceAfter=6))
    styles.add(ParagraphStyle(name='NormalCenter', alignment=1))
    
    elements = []
    
    # Logo and Company Info
    logo = Image(logo_path, 1.5 * inch, 1.5 * inch)
    company_info = [
        Paragraph("YerikoDJ-bookings", styles['TitleCenter']),
        Paragraph("Email: ygvargas93@gmail.com", styles['Normal']),
        Paragraph("Phone Number: (646) 771-6111", styles['Normal'])
    ]
    company_table = Table([[company_info, logo]], colWidths=[4.5 * inch, 1.5 * inch])
    company_table.setStyle(TableStyle([
        ('ALIGN', (0, 0), (0, 0), 'LEFT'),
        ('ALIGN', (1, 0), (1, 0), 'RIGHT')
    ]))
    elements.append(company_table)
    elements.append(Spacer(1, 0.5 * inch))
    
    # Invoice Header
    elements.append(Paragraph(f"Invoice #: {invoice_number}", styles['Header']))
    elements.append(Paragraph(f"Invoice Date: {invoice_date}", styles['Normal']))
    elements.append(Paragraph(f"Due Date: {due_date}", styles['Normal']))
    elements.append(Spacer(1, 0.25 * inch))
    elements.append(HRFlowable(width="100%", thickness=1, color=colors.black))
    elements.append(Spacer(1, 0.25 * inch))
    
    # Billing Info
    elements.append(Paragraph("Bill To:", styles['Header']))
    elements.append(Paragraph(recipient_name, styles['Normal']))
    elements.append(Paragraph(f"Phone: {recipient_phone}", styles['Normal']))
    elements.append(Paragraph(f"Email: {recipient_email}", styles['Normal']))
    elements.append(Spacer(1, 0.5 * inch))
    
    # Service Description Table (Includes hourly rate calculation and optional speaker rental)
    service_data = [["Date", "Service", "Time Slot", "Rate", "Total"]]
    service_data.append([service_date, dj_name, time_slot, f"${float(rate):.2f}/hr", f"${base_rate:.2f}"])
    if speaker_rental > 0:
        service_data.append([service_date, "Speaker Rental", "", f"${speaker_rental:.2f}", f"${speaker_rental:.2f}"])
    service_table = Table(service_data, colWidths=[1.5 * inch] * 5)
    service_table.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.black),
        ('TEXTCOLOR', (0, 0), (-1, 0), colors.white),
        ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
        ('GRID', (0, 0), (-1, -1), 1, colors.black)
    ]))
    elements.append(service_table)
    elements.append(Spacer(1, 0.5 * inch))
    
    # Financial Summary Table (Includes speaker rental, if any)
    summary_data = [
        ["Base Rate:", f"${base_rate:.2f}"],
        ["Additional:", f"${additional_amount:.2f}"],
        ["Service Fee:", f"${service_fee:.2f}"]
    ]
    if speaker_rental > 0:
        summary_data.append(["Speaker Rental:", f"${speaker_rental:.2f}"])
    summary_data.append(["Total Due:", f"${total_due_amount:.2f}"])
    summary_table = Table(summary_data, colWidths=[3 * inch, 2 * inch])
    summary_table.setStyle(TableStyle([
        ('TEXTCOLOR', (0, 0), (-1, -2), colors.black),
        ('BACKGROUND', (-1, -1), (-1, -1), colors.lightgrey),
        ('GRID', (0, 0), (-1, -1), 1, colors.black)
    ]))
    elements.append(summary_table)
    elements.append(Spacer(1, 0.5 * inch))
    
    # Payment Instructions
    elements.append(Paragraph("Payment Instructions:", styles['Header']))
    elements.append(Paragraph(f"Venmo: {venmo_handle}", styles['Normal']))
    elements.append(Paragraph(f"Cash App: {cashapp_handle}", styles['Normal']))
    
    # Payment Plan (if an advance payment is required)
    if advance_payment > 0:
        remaining_payment = total_due_amount - advance_payment
        elements.append(Spacer(1, 0.25 * inch))
        elements.append(Paragraph("Payment Plan:", styles['Header']))
        elements.append(Paragraph(f"Advance Payment: ${advance_payment:.2f} (Due by: {due_date})", styles['Normal']))
        elements.append(Paragraph(f"Remaining Payment: ${remaining_payment:.2f} (Due by: {final_payment_due_date})", styles['Normal']))
        elements.append(Spacer(1, 0.5 * inch))
    
    # Payment Image
    bw_image_path = convert_image_to_bw(payment_image_path)
    payment_image = Image(bw_image_path, 3 * inch, 2 * inch)
    elements.append(payment_image)
    elements.append(Spacer(1, 0.5 * inch))
    
    # Thank You note
    elements.append(Paragraph("paypal.me/yerikovargas ... Thank you!", styles['TitleCenter']))
    
    # Build PDF
    doc.build(elements)
    print(f"Invoice saved as {pdf_filename}")


In [4]:

#!#!#!#!#! RUNNING STATEMENTS #!#!#!#!#!
create_invoice(
    invoice_number="ac-25-53",
    invoice_date="05/30/2025",         # Today's date
    service_date="05/26/2025",           # Service date on May 10
    recipient_name="Puma",
    recipient_phone="",  # Update with actual phone number
    recipient_email="",                  # Update with actual email
    dj_name="YerikoDJ",
    time_slot="6 PM - 11 PM",
    rate="62.1",                         # Hourly rate in dollars
    duration_hours=0,                   # Calculated as 5 hours (6-11 PM)
    additional_amount=0,                # No additional charges
    service_fee=62,                      # No service fee
    due_date="06/10/2025",              # Due date for payment if needed
    venmo_handle="@lizardini",
    cashapp_handle="@lizardini",
    payment_image_path="pics/method_paym.jpg",
    logo_path="pics/logo.png",
    speaker_rental=0,                 # Speaker rental fee is $160
    advance_payment=0,                  # No advance payment in this deal
    final_payment_due_date="06/10/2025"           # No final payment date needed here
)


Invoice saved as invoice_ac-25-53.pdf
